# Laboratori 2 — Kompleksiteti Kohor dhe Merge Sort

## 1. Teoria

### Çfarë është Kompleksiteti Kohor?

Kompleksiteti kohor tregon se sa operacione bën një algoritëm në varësi të madhësisë së të dhënave (n).

> **Shënim:** Nuk matim kohën në sekonda (që varet nga kompjuteri), por **numrin e operacioneve**.

### Notacioni Big O — O(n)

Big O përshkruan sa shpejt rritet numri i operacioneve kur n bëhet shumë i madh.

| Notacion | Emërtimi | Shembull |
|---|---|---|
| O(1) | Konstant | Lexo elementin e parë të listës |
| O(log n) | Logaritmik | Binary Search |
| O(n) | Linear | Kërko elementin në listë |
| O(n log n) | Linearitmik | Merge Sort, Quick Sort |
| O(n²) | Kuadratik | Insertion Sort, Selection Sort |
| O(2ⁿ) | Eksponencial | Problemet rekursive naive |

### Rregulla e artë: n₀

- Për **n të vogla** (n < n₀): kompleksiteti nuk ka rëndësi të madhe
- Për **n të mëdha** (n > n₀): kompleksiteti bëhet kritik

**Pyetja kryesore:** A mund të bëjmë më mirë se O(n²)?
**Përgjigjja:** PO — Merge Sort arrin O(n log n)!

---

## Merge Sort — Renditja me Bashkim

**Ideja:** Përca e Sundo *(Divide and Conquer)*

1. **Përca** listën në dy gjysma
2. **Rëndit** secilën gjysmë (rekursivisht)
3. **Bashko** dy gjysmat e renditura

**Kompleksiteti:** O(n log n) — gjithmonë, pavarësisht nga rendi fillestar.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

n_small = np.linspace(0.1, 5, 200)
n_large = np.linspace(1, 100, 500)

funksionet = [
    ("O(1)",       lambda n: np.ones_like(n),      "#3498db"),
    ("O(log n)",   lambda n: np.log2(n),           "#2ecc71"),
    ("O(n)",       lambda n: n,                    "#f39c12"),
    ("O(n log n)", lambda n: n * np.log2(n),       "#9b59b6"),
    ("O(n²)",      lambda n: n**2,                 "#e74c3c"),
]

fig = make_subplots(rows=1, cols=2,
    subplot_titles=(
        "n të vogla (0–5)",
        "n të mëdha (1–100) — O(n²) vs O(2ⁿ)"
    ))

for name, f, color in funksionet:
    y = f(n_small)
    fig.add_trace(go.Scatter(
        x=n_small, y=y, name=name,
        line=dict(color=color, width=2),
        text=[f"n={x:.2f}<br>{name} = {v:.2f}" for x, v in zip(n_small, y)],
        hovertemplate="%{text}<extra></extra>",
        showlegend=True
    ), row=1, col=1)

for name, f, color in funksionet:
    y = f(n_large)
    fig.add_trace(go.Scatter(
        x=n_large, y=y, name=name,
        line=dict(color=color, width=2),
        text=[f"n={x:.1f}<br>{name} = {v:.1f}" for x, v in zip(n_large, y)],
        hovertemplate="%{text}<extra></extra>",
        showlegend=False
    ), row=1, col=2)

n_exp = np.arange(1, 18)
fig.add_trace(go.Scatter(
    x=n_exp, y=2**n_exp, name="O(2ⁿ)",
    line=dict(color="#1abc9c", width=2, dash="dot"),
    text=[f"n={n}<br>O(2ⁿ) = {2**n:,}" for n in n_exp],
    hovertemplate="%{text}<extra></extra>",
    showlegend=False
), row=1, col=2)

fig.update_layout(
    height=500,
    width=1100,
    margin=dict(l=60, r=20, t=80, b=60),
    title="Krahasimi i Funksioneve — hover mbi funksione për vlerat ekzakte",
    hoverlabel=dict(bgcolor="#1e1e2e", font_color="white", font_size=11),
    plot_bgcolor="white",
    hovermode="x",
    legend=dict(x=0.5, y=-0.15, orientation="h", xanchor="center")
)
fig.update_yaxes(title_text="Vlera e funksionit f(n)", gridcolor="#eeeeee", range=[0, 10], row=1, col=1)
fig.update_yaxes(title_text="Vlera e funksionit f(n)", gridcolor="#eeeeee", range=[0, 10000], row=1, col=2)
fig.update_xaxes(title_text="n", gridcolor="#eeeeee", range=[0, 5], dtick=1, row=1, col=1)
fig.update_xaxes(title_text="n", gridcolor="#eeeeee", row=1, col=2)
fig.show()

In [ ]:
import pygame
import random
import os

pygame.init()

screen = pygame.display.set_mode((1280, 780), pygame.RESIZABLE)
pygame.display.set_caption("Merge Sort")

font_medium = pygame.font.Font(None, 30)
font_small  = pygame.font.Font(None, 22)

BG        = (18, 18, 30)
TEXT_COL  = (220, 220, 240)
ACCENT    = (100, 100, 160)
PANEL_COL = (28, 28, 45)
C_UNSORT  = (60, 60, 90)
C_ACTIVE  = (255, 99, 99)
C_COMPARE = (255, 220, 50)
C_MERGED  = (72, 199, 142)
C_DONE    = (72, 199, 142)

raw = input("Shkruaj numrat (ose Enter për listë rastësore): ").strip()
try:    lst = [int(x) for x in raw.split()] if raw else random.sample(range(1, 51), 16)
except: lst = random.sample(range(1, 51), 16)
original = lst.copy()

steps = []

def build_steps(arr):
    state = arr.copy()
    steps.append(("start", state.copy(), -1, -1, [],
                  "Lista fillestare — do të ndahet në gjysmë rekursivisht"))

    def rec(left, right):
        if right - left <= 1:
            return
        mid = (left + right) // 2
        steps.append(("divide", state.copy(), left, right, list(range(left, right)),
                      f"Ndaj [{left}..{right-1}] → [{left}..{mid-1}] dhe [{mid}..{right-1}]"))
        rec(left, mid)
        rec(mid, right)

        i, j, tmp = left, mid, []
        while i < mid and j < right:
            steps.append(("compare", state.copy(), i, j, list(range(left, right)),
                          f"Krahaso {state[i]} (i={i}) me {state[j]} (i={j})"))
            if state[i] <= state[j]:
                tmp.append(state[i]); i += 1
            else:
                tmp.append(state[j]); j += 1
        tmp.extend(state[i:mid])
        tmp.extend(state[j:right])
        for k, v in enumerate(tmp):
            state[left + k] = v
        steps.append(("merge", state.copy(), left, right, list(range(left, right)),
                      f"Bashko [{left}..{mid-1}] + [{mid}..{right-1}] → {tmp}"))

    rec(0, len(arr))
    steps.append(("done", state.copy(), -1, -1, [],
                  "✓  E RENDITUR!   Shtyp R për rivendosje"))

build_steps(lst.copy())

step_idx = [0]
msg = steps[0][5]

def draw():
    W, H = screen.get_size()
    screen.fill(BG)
    kind, state, ia, ib, active, _ = steps[step_idx[0]]
    n   = len(state)
    P   = 50
    CT  = 80
    CB  = H - 150
    CW  = W - 2 * P
    CH  = CB - CT
    bw  = min(120, CW // n)
    gap = max(4, bw // 6)
    sx  = P + (CW - n * bw) // 2
    lo, hi = min(state), max(state)
    scale = (CH - 40) / (hi - lo) if hi > lo else CH - 40

    pygame.draw.rect(screen, PANEL_COL, (P-15, CT-15, CW+30, CH+30), border_radius=12)

    for idx, val in enumerate(state):
        x  = sx + idx * bw + gap // 2
        bh = int((val - lo) * scale) + 30
        y  = CB - bh
        w  = bw - gap
        if kind == "done":
            color = C_DONE
        elif idx == ia and kind == "compare":
            color = C_ACTIVE
        elif idx == ib and kind == "compare":
            color = C_COMPARE
        elif idx in active and kind == "merge":
            color = C_MERGED
        elif idx in active:
            color = C_ACTIVE
        else:
            color = C_UNSORT
        pygame.draw.rect(screen, (10,10,20), (x+3, y+3, w, bh), border_radius=6)
        pygame.draw.rect(screen, color,      (x,   y,   w, bh), border_radius=6)
        pygame.draw.rect(screen, tuple(min(255, c+60) for c in color), (x+4, y+2, w-8, 4), border_radius=3)
        t = font_small.render(str(val), True, TEXT_COL)
        screen.blit(t, (x + w//2 - t.get_width()//2, y - 18))

    lx = P
    for label, col in [("Aktiv/Ndan", C_ACTIVE), ("Krahason", C_COMPARE), ("Bashkuar", C_MERGED), ("Pa renditur", C_UNSORT)]:
        pygame.draw.rect(screen, col, (lx, 22, 24, 24), border_radius=5)
        t = font_small.render(label, True, TEXT_COL)
        screen.blit(t, (lx + 32, 26))
        lx += t.get_width() + 60

    pygame.draw.rect(screen, PANEL_COL, (P-15, CB+15, CW+30, 120), border_radius=12)
    screen.blit(font_small.render("Lista:  [" + ", ".join(map(str, state)) + "]", True, TEXT_COL), (P, CB+28))
    screen.blit(font_small.render(f"Hapi {step_idx[0] + 1} / {len(steps)}", True, ACCENT), (P, CB+50))
    screen.blit(font_medium.render(msg, True, TEXT_COL), (P, CB+72))
    screen.blit(font_small.render("SPACE: hapi tjetër   |   R: rivendos", True, ACCENT), (P, H-25))
    pygame.display.flip()

def step():
    global msg
    if step_idx[0] < len(steps) - 1:
        step_idx[0] += 1
    msg = steps[step_idx[0]][5]

def reset():
    global msg
    step_idx[0] = 0
    msg = steps[0][5]

draw()
clock   = pygame.time.Clock()
running = True
while running:
    clock.tick(60)
    for e in pygame.event.get():
        if e.type == pygame.QUIT:
            pygame.quit()
            os._exit(0)
        elif e.type == pygame.VIDEORESIZE: draw()
        elif e.type == pygame.KEYDOWN:
            if   e.key == pygame.K_SPACE: step(); draw()
            elif e.key == pygame.K_r:     reset(); draw()

pygame 2.6.1 (SDL 2.28.4, Python 3.9.6)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [ ]:
import time
import random
import plotly.graph_objects as go


def insertion_sort(lista):
    n = len(lista)
    for i in range(1, n):
        vlera = lista[i]
        j = i - 1
        while j >= 0 and lista[j] > vlera:
            lista[j + 1] = lista[j]
            j -= 1
        lista[j + 1] = vlera
    return lista


def selection_sort(lista):
    n = len(lista)
    for i in range(n - 1):
        min_idx = i
        for j in range(i + 1, n):
            if lista[j] < lista[min_idx]:
                min_idx = j
        lista[i], lista[min_idx] = lista[min_idx], lista[i]
    return lista


def merge_sort(lista):
    if len(lista) <= 1:
        return lista
    mid = len(lista) // 2
    left  = merge_sort(lista[:mid])
    right = merge_sort(lista[mid:])
    res, i, j = [], 0, 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]: res.append(left[i]); i += 1
        else:                   res.append(right[j]); j += 1
    res.extend(left[i:]); res.extend(right[j:])
    return res


def mat_kohen(funksioni, lista):
    c = lista.copy()
    t = time.perf_counter()
    funksioni(c)
    return time.perf_counter() - t


madhesite = [10, 50, 100, 500, 1000, 2000, 5000]
raste     = ["Random", "Sorted", "Reverse"]
colors    = {"Insertion Sort": "#3498db", "Selection Sort": "#e74c3c", "Merge Sort": "#2ecc71"}

rezultatet = {alg: {r: [] for r in raste} for alg in colors}

for n in madhesite:
    listat = {
        "Random":  random.sample(range(n * 10), n),
        "Sorted":  list(range(n)),
        "Reverse": list(range(n, 0, -1)),
    }
    for rast, lst in listat.items():
        rezultatet["Insertion Sort"][rast].append(mat_kohen(insertion_sort, lst))
        rezultatet["Selection Sort"][rast].append(mat_kohen(selection_sort, lst))
        rezultatet["Merge Sort"][rast].append(mat_kohen(merge_sort, lst))

fig = go.Figure()

for alg, color in colors.items():
    for rast in raste:
        kohet = rezultatet[alg][rast]
        ht    = [f"{alg}<br>Rasti: {rast}<br>n={n}<br>Koha: {k*1000:.4f} ms"
                 for n, k in zip(madhesite, kohet)]
        dash  = "solid" if rast == "Random" else ("dot" if rast == "Sorted" else "dash")
        fig.add_trace(go.Scatter(
            x=madhesite, y=kohet,
            name=f"{alg} — {rast}",
            line=dict(color=color, width=2, dash=dash),
            mode="lines+markers", marker=dict(size=7),
            text=ht, hovertemplate="%{text}<extra></extra>"
        ))

fig.update_layout(
    title="Insertion vs Selection vs Merge Sort — të gjitha rastet",
    xaxis=dict(title="n (madhësia e listës)"),
    yaxis=dict(title="Koha (sekonda)"),
    hoverlabel=dict(bgcolor="#1e1e2e", font_color="white", font_size=11),
    plot_bgcolor="white",
    legend=dict(groupclick="toggleitem")
)
fig.update_yaxes(gridcolor="#eeeeee")
fig.update_xaxes(gridcolor="#eeeeee")
fig.show()

In [ ]:
import time
import random
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def insertion_sort(lista):
    n = len(lista)
    for i in range(1, n):
        vlera = lista[i]
        j = i - 1
        while j >= 0 and lista[j] > vlera:
            lista[j + 1] = lista[j]
            j -= 1
        lista[j + 1] = vlera
    return lista


def merge_sort(lista):
    if len(lista) <= 1:
        return lista
    mid = len(lista) // 2
    left  = merge_sort(lista[:mid])
    right = merge_sort(lista[mid:])
    res, i, j = [], 0, 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]: res.append(left[i]); i += 1
        else:                   res.append(right[j]); j += 1
    res.extend(left[i:]); res.extend(right[j:])
    return res


def mat_kohen(funksioni, lista):
    c = lista.copy()
    t = time.perf_counter()
    funksioni(c)
    return time.perf_counter() - t


n_test   = list(range(10, 301, 10))
ins_koha = []
mrg_koha = []

for n in n_test:
    lst     = random.sample(range(n * 10), n)
    ins_koha.append(mat_kohen(insertion_sort, lst))
    mrg_koha.append(mat_kohen(merge_sort, lst))

raporti = [ins / mrg if mrg > 0 else 1 for ins, mrg in zip(ins_koha, mrg_koha)]

n0 = next((n_test[i] for i, r in enumerate(raporti) if r >= 2), None)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=(
        "Koha absolute: Insertion vs Merge Sort",
        f"Sa herë më i ngadaltë është Insertion?  n₀ ≈ {n0}"
    ))

fig.add_trace(go.Scatter(x=n_test, y=ins_koha, name="Insertion Sort",
    line=dict(color="#3498db", width=2), mode="lines+markers",
    text=[f"Insertion Sort<br>n={n}<br>{k*1000:.4f} ms" for n, k in zip(n_test, ins_koha)],
    hovertemplate="%{text}<extra></extra>"), row=1, col=1)

fig.add_trace(go.Scatter(x=n_test, y=mrg_koha, name="Merge Sort",
    line=dict(color="#2ecc71", width=2), mode="lines+markers",
    text=[f"Merge Sort<br>n={n}<br>{k*1000:.4f} ms" for n, k in zip(n_test, mrg_koha)],
    hovertemplate="%{text}<extra></extra>"), row=1, col=1)

fig.add_trace(go.Scatter(x=n_test, y=raporti, name="Raporti",
    line=dict(color="#e74c3c", width=2), mode="lines+markers",
    text=[f"n={n}<br>Insertion është {r:.1f}× më i ngadaltë" for n, r in zip(n_test, raporti)],
    hovertemplate="%{text}<extra></extra>", showlegend=False), row=1, col=2)

fig.add_hline(y=1, line_dash="dot", line_color="green",
    annotation_text="Insertion = Merge", row=1, col=2)
fig.add_hline(y=2, line_dash="dot", line_color="orange",
    annotation_text="Insertion = 2× Merge", row=1, col=2)
if n0:
    fig.add_vline(x=n0, line_dash="dash", line_color="red",
        annotation_text=f"n₀ ≈ {n0}", row=1, col=2)

fig.update_layout(
    height=480,
    title=f"Ku bëhet Merge Sort më i dobishëm? — n₀ ≈ {n0}",
    hoverlabel=dict(bgcolor="#1e1e2e", font_color="white", font_size=11),
    plot_bgcolor="white"
)
fig.update_yaxes(gridcolor="#eeeeee")
fig.update_xaxes(gridcolor="#eeeeee")
fig.show()

print(f"n₀ ≈ {n0}")
print(f"Për n < {n0}:  Insertion dhe Merge janë gati njësoj")
print(f"Për n > {n0}:  Merge Sort bëhet dukshëm më i shpejtë")
print(f"Për n = 300:  Insertion është {raporti[-1]:.1f}× më i ngadaltë se Merge Sort")

n₀ ≈ 160
Për n < 160:  Insertion dhe Merge janë gati njësoj
Për n > 160:  Merge Sort bëhet dukshëm më i shpejtë
Për n = 300:  Insertion është 3.5× më i ngadaltë se Merge Sort


In [ ]:
import time
import random
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def insertion_sort(lista):
    n = len(lista)
    for i in range(1, n):
        vlera = lista[i]
        j = i - 1
        while j >= 0 and lista[j] > vlera:
            lista[j + 1] = lista[j]
            j -= 1
        lista[j + 1] = vlera
    return lista

def selection_sort(lista):
    n = len(lista)
    for i in range(n - 1):
        min_idx = i
        for j in range(i + 1, n):
            if lista[j] < lista[min_idx]:
                min_idx = j
        lista[i], lista[min_idx] = lista[min_idx], lista[i]
    return lista

def merge_sort(lista):
    if len(lista) <= 1:
        return lista
    mid = len(lista) // 2
    left  = merge_sort(lista[:mid])
    right = merge_sort(lista[mid:])
    res, i, j = [], 0, 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]: res.append(left[i]); i += 1
        else:                   res.append(right[j]); j += 1
    res.extend(left[i:]); res.extend(right[j:])
    return res

def mat_kohen(funksioni, lista):
    c = lista.copy()
    t = time.perf_counter()
    funksioni(c)
    return time.perf_counter() - t

madhesite = [10, 100, 1000, 10000]
raste     = ['Random', 'Sorted', 'Reverse']
algoritme = [
    ("Insertion Sort", insertion_sort, "#3498db"),
    ("Selection Sort", selection_sort, "#e74c3c"),
    ("Merge Sort",     merge_sort,     "#2ecc71"),
]

fig = make_subplots(rows=2, cols=2,
    subplot_titles=[f"n = {n}" for n in madhesite])

positions = [(1,1), (1,2), (2,1), (2,2)]

for idx, n in enumerate(madhesite):
    row, col = positions[idx]
    lista_random  = [7,3,9,2,5,8,1,6,4,10] if n==10 else random.sample(range(1, n+1), n)
    lista_sorted  = list(range(1, n+1))
    lista_reverse = list(range(n, 0, -1))
    listat = [lista_random, lista_sorted, lista_reverse]

    for em, alg, color in algoritme:
        kohet = [mat_kohen(alg, l) for l in listat]
        fig.add_trace(go.Scatter(
            x=raste, y=kohet,
            name=em, showlegend=(idx == 0),
            mode="lines+markers",
            line=dict(color=color, width=2),
            marker=dict(size=8),
            text=[f"{em}<br>{r}: {k:.8f} sek" for r, k in zip(raste, kohet)],
            hovertemplate="%{text}<extra></extra>"
        ), row=row, col=col)

fig.update_layout(
    height=700, width=1000,
    title="Insertion vs Selection vs Merge Sort",
    hoverlabel=dict(bgcolor="#1e1e2e", font_color="white", font_size=11),
    plot_bgcolor="white",
    legend=dict(x=0.5, y=-0.08, orientation="h", xanchor="center")
)
fig.update_yaxes(title_text="Koha (sek)", gridcolor="#eeeeee")
fig.update_xaxes(gridcolor="#eeeeee")
fig.show()